# Setup

In [ ]:
!pip install -qU anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.4/243.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 600.5/600.5 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.0.2 which is incompatible.


Ten kod wykonuje instalację dwóch pakietów Python poprzez narzędzie pip:

1. `anthropic` - to oficjalna biblioteka kliencka firmy Anthropic, która umożliwia programistom integrację z modelami językowymi Claude. Dzięki tej bibliotece możesz wysyłać zapytania do API Claude i otrzymywać odpowiedzi.

In [2]:
from anthropic import Anthropic
import re
from google.colab import userdata

In [ ]:
class CFG:
    model = "claude-3-haiku-20240307"
    max_tokens = 2048

W tej klasie zdefiniowane są dwa atrybuty:

1. `model = "claude-3-haiku-20240307"`
   Ten atrybut określa konkretną wersję modelu Claude, która ma być używana. "Claude-3-Haiku" to jeden z modeli z rodziny Claude 3, zaprojektowany z myślą o szybkości działania. Suffix "20240307" wskazuje na datę wydania modelu (7 marca 2024), co jest istotne, ponieważ modele są aktualizowane i rozwijane w czasie.

2. `max_tokens = 2048`
   Ten atrybut określa maksymalną liczbę tokenów, jakie model może wygenerować w odpowiedzi. Token to podstawowa jednostka tekstu używana przez modele językowe - może to być część słowa, całe słowo lub znak interpunkcyjny. Wartość 2048 oznacza, że model będzie generował odpowiedzi o długości do 2048 tokenów, co przekłada się na około 1500-2000 słów, zależnie od języka i złożoności tekstu.

In [4]:
client = Anthropic(api_key = userdata.get('claude'))

# Funkcje

In [5]:
def calculate(expression):
    expression = re.sub(r'[^0-9+\-*/().]', '', expression)

    try:
        result = eval(expression)
        return str(result)
    except (SyntaxError, ZeroDivisionError, NameError, TypeError, OverflowError):
        return "Error: Invalid expression"

W pierwszej linii wewnątrz funkcji znajduje się wyrażenie regularne:
```python
expression = re.sub(r'[^0-9+\-*/().]', '', expression)
```

Ta linia filtruje wyrażenie, usuwając z niego wszystkie znaki oprócz:
- cyfr (0-9)
- operatorów matematycznych (+, -, *, /)
- nawiasów ()
- kropki dziesiętnej (.)

Jest to mechanizm zabezpieczający, który zapobiega wykonaniu niebezpiecznego kodu przez funkcję `eval()`. Na przykład, jeśli ktoś wprowadzi `"1+1; import os; os.system('rm -rf /')"`, to po przefiltrowaniu zostanie tylko `"1+1"`, a reszta kodu zostanie usunięta.

In [ ]:
def process_tool_call(tool_name, tool_input):
    if tool_name == "calculator":
        return calculate(tool_input["expression"])

Ta funkcja `process_tool_call` służy jako pośrednik między zapytaniem użytkownika a odpowiednimi narzędziami, które mogą obsługiwać różne rodzaje żądań. Jest to przykład wzorca projektowego znanego jako "router" lub "dispatcher".

In [7]:
def chat_with_claude_no_tool(user_message):

    message = client.messages.create(
        model=CFG.model, max_tokens= CFG.max_tokens, messages=[{"role": "user", "content": user_message}],
         )

    print(f"\nInitial Response:")
    print(f"Stop Reason: {message.stop_reason}")
    print(f"Content: {message.content}")

    response = message

    final_response = next(
        (block.text for block in response.content if hasattr(block, "text")),
        None,
    )
    print(response.content)
    print(f"\nFinal Response: {final_response}")

    return final_response

Ta funkcja `chat_with_claude_no_tool` służy do komunikacji z modelem Claude bez wykorzystania narzędzi pomocniczych. Funkcja przyjmuje wiadomość od użytkownika i zwraca odpowiedź od modelu. Przeanalizujmy ją krok po kroku:

1. Wyodrębnianie właściwego tekstu odpowiedzi:
```python
final_response = next(
    (block.text for block in response.content if hasattr(block, "text")),
    None,
)
```
Ta konstrukcja służy do wydobycia tekstu z odpowiedzi. Odpowiedź z API może zawierać różne typy bloków (tekst, obrazy, itp.). Funkcja:
- iteruje przez wszystkie bloki w `response.content`
- sprawdza, które bloki mają atrybut `text` (czyli są blokami tekstowymi)
- używa funkcji `next()` aby zwrócić pierwszy znaleziony tekst ITERATOR
- jeśli nie znajdzie żadnego tekstu, zwraca `None`

In [8]:
def chat_with_claude_tool(user_message):

    message = client.messages.create(
        model=CFG.model, max_tokens= CFG.max_tokens, messages=[{"role": "user", "content": user_message}],
        tools=tools,)

    print(f"\nInitial Response:")
    print(f"Stop Reason: {message.stop_reason}")
    print(f"Content: {message.content}")

    if message.stop_reason == "tool_use":
        tool_use = next(block for block in message.content if block.type == "tool_use")
        tool_name = tool_use.name
        tool_input = tool_use.input

        print(f"\nTool Used: {tool_name}")
        print(f"Tool Input: {tool_input}")

        tool_result = process_tool_call(tool_name, tool_input)

        print(f"Tool Result: {tool_result}")

        response = client.messages.create(
            model=CFG.model,
            max_tokens = CFG.max_tokens,
            messages=[
                {"role": "user", "content": user_message},
                {"role": "assistant", "content": message.content},
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "tool_result",
                            "tool_use_id": tool_use.id,
                            "content": tool_result,
                        }
                    ],
                },
            ],
            tools=tools,
        )
    else:
        response = message

    final_response = next(
        (block.text for block in response.content if hasattr(block, "text")),
        None,
    )
    print(response.content)
    print(f"\nFinal Response: {final_response}")

    return final_response

`chat_with_claude_tool` umożliwia komunikację z modelem Claude z obsługą narzędzi pomocniczych. Jest to bardziej zaawansowana wersja funkcji `chat_with_claude_no_tool`, która potrafi rozpoznać, kiedy model chce użyć narzędzia, wykonać obliczenia za pomocą tego narzędzia, a następnie kontynuować rozmowę z modelem, uwzględniając wyniki tych obliczeń.

# Test

In [9]:
tools = [
    {
        "name": "calculator",
        "description": "A simple calculator that performs basic arithmetic operations.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "The mathematical expression to evaluate (e.g., '2 + 3 * 4')."
                }
            },
            "required": ["expression"]
        }
    }
]

Zmienna `tools` to lista zawierająca definicję narzędzi, które mogą być wykorzystane przez model Claude. W tym konkretnym przypadku zdefiniowane jest tylko jedno narzędzie - kalkulator.

Ta struktura danych jest kluczowym elementem umożliwiającym modelowi rozpoznanie, kiedy powinien użyć zewnętrznego narzędzia zamiast próbować samodzielnie rozwiązać problem. Przyjrzyjmy się jej składowym:

`tools` to lista obiektów, gdzie każdy obiekt reprezentuje jedno narzędzie i składa się z trzech głównych elementów:

1. `"name": "calculator"` - Jest to identyfikator narzędzia. Kiedy model decyduje się użyć kalkulatora, odwołuje się do niego używając tej nazwy. Ta sama nazwa jest sprawdzana w funkcji `process_tool_call`, aby wybrać odpowiednią funkcję do wykonania.

2. `"description": "A simple calculator that performs basic arithmetic operations."` - Ten opis informuje model, do czego służy narzędzie. Model używa tego opisu, aby zdecydować, czy dane narzędzie jest odpowiednie do rozwiązania konkretnego problemu przedstawionego przez użytkownika.

3. `"input_schema"` - Ta część definiuje strukturę danych, jakiej narzędzie oczekuje. Jest to schemat w formacie JSON Schema, który:
   - Określa, że dane wejściowe muszą być obiektem (`"type": "object"`)
   - Definiuje właściwości tego obiektu (`"properties"`)
   - Wskazuje, że obiekt musi mieć właściwość o nazwie "expression" (`"required": ["expression"]`)
   
   W przypadku właściwości "expression":
   - Jest to ciąg znaków (`"type": "string"`)
   - Ma opis wyjaśniający, co powinno się w nim znaleźć (`"description": "The mathematical expression to evaluate (e.g., '2 + 3 * 4')."`)

Ta struktura danych pełni rolę kontraktu między modelem a kodem obsługującym narzędzia. Dzięki precyzyjnej definicji:
- Model wie, kiedy i jak używać kalkulatora
- Model wie, jaki format danych powinien przekazać
- Kod obsługujący narzędzia wie, czego oczekiwać od modelu

W praktycznym zastosowaniu, gdy użytkownik zada pytanie wymagające obliczeń matematycznych, model może zdecydować, że potrzebuje użyć kalkulatora. Wtedy zatrzymuje generowanie odpowiedzi, wysyła żądanie użycia narzędzia z odpowiednimi parametrami, a system wywołuje funkcję `calculate` z podanym wyrażeniem. Wynik zostaje następnie przekazany z powrotem do modelu, który może kontynuować tworzenie odpowiedzi, mając już dokładny wynik obliczeń.

In [13]:
question1 = "What is the result of 1,984,135 * 9,343,116?"

In [22]:
chat_with_claude_no_tool(question1)


Initial Response:
Stop Reason: end_turn
Content: [TextBlock(citations=None, text="Okay, let's solve this step-by-step:\n1) 1,984,135 * 9,343,116\n2) To multiply these two numbers, we need to multiply each digit in the first number with each digit in the second number.\n3) 1,984,135 * 9,343,116 = (1 * 9,343,116) + (9 * 9,343,116) + (8 * 9,343,116) + (4 * 9,343,116) + (1 * 9,343,116) + (3 * 9,343,116) + (5 * 9,343,116)\n4) Calculating each part:\n1 * 9,343,116 = 9,343,116\n9 * 9,343,116 = 84,088,044\n8 * 9,343,116 = 74,744,928\n4 * 9,343,116 = 37,372,464\n1 * 9,343,116 = 9,343,116\n3 * 9,343,116 = 28,029,348\n5 * 9,343,116 = 46,715,580\n5) Adding all the parts together:\n9,343,116 + 84,088,044 + 74,744,928 + 37,372,464 + 9,343,116 + 28,029,348 + 46,715,580 = 289,636,596\n\nTherefore, the result of 1,984,135 * 9,343,116 is 289,636,596.", type='text')]
[TextBlock(citations=None, text="Okay, let's solve this step-by-step:\n1) 1,984,135 * 9,343,116\n2) To multiply these two numbers, we need

"Okay, let's solve this step-by-step:\n1) 1,984,135 * 9,343,116\n2) To multiply these two numbers, we need to multiply each digit in the first number with each digit in the second number.\n3) 1,984,135 * 9,343,116 = (1 * 9,343,116) + (9 * 9,343,116) + (8 * 9,343,116) + (4 * 9,343,116) + (1 * 9,343,116) + (3 * 9,343,116) + (5 * 9,343,116)\n4) Calculating each part:\n1 * 9,343,116 = 9,343,116\n9 * 9,343,116 = 84,088,044\n8 * 9,343,116 = 74,744,928\n4 * 9,343,116 = 37,372,464\n1 * 9,343,116 = 9,343,116\n3 * 9,343,116 = 28,029,348\n5 * 9,343,116 = 46,715,580\n5) Adding all the parts together:\n9,343,116 + 84,088,044 + 74,744,928 + 37,372,464 + 9,343,116 + 28,029,348 + 46,715,580 = 289,636,596\n\nTherefore, the result of 1,984,135 * 9,343,116 is 289,636,596."

In [23]:
chat_with_claude_tool(question1)


Initial Response:
Stop Reason: tool_use
Content: [TextBlock(citations=None, text="Okay, let's calculate the result of 1,984,135 * 9,343,116 using the calculator tool:", type='text'), ToolUseBlock(id='toolu_012PQ6ZGBTVw8FMb2Hyy7GPL', input={'expression': '1984135 * 9343116'}, name='calculator', type='tool_use')]

Tool Used: calculator
Tool Input: {'expression': '1984135 * 9343116'}
Tool Result: 18538003464660
[TextBlock(citations=None, text='The result of 1,984,135 * 9,343,116 is 18,538,003,464,660.', type='text')]

Final Response: The result of 1,984,135 * 9,343,116 is 18,538,003,464,660.


'The result of 1,984,135 * 9,343,116 is 18,538,003,464,660.'

In [34]:
eval('1984135 * 9343116')

18538003464660

In [16]:
question2 = "Calculate (12851 - 593) * 301 + 76"

In [17]:
chat_with_claude_no_tool(question2)


Initial Response:
Stop Reason: end_turn
Content: [TextBlock(citations=None, text="Okay, let's calculate (12851 - 593) * 301 + 76.\n\nFirst, let's calculate the difference between 12851 and 593:\n12851 - 593 = 12258\n\nNow, let's multiply 12258 by 301:\n12258 * 301 = 3,685,758\n\nFinally, let's add 76 to the result:\n3,685,758 + 76 = 3,685,834\n\nTherefore, the result of (12851 - 593) * 301 + 76 is 3,685,834.", type='text')]
[TextBlock(citations=None, text="Okay, let's calculate (12851 - 593) * 301 + 76.\n\nFirst, let's calculate the difference between 12851 and 593:\n12851 - 593 = 12258\n\nNow, let's multiply 12258 by 301:\n12258 * 301 = 3,685,758\n\nFinally, let's add 76 to the result:\n3,685,758 + 76 = 3,685,834\n\nTherefore, the result of (12851 - 593) * 301 + 76 is 3,685,834.", type='text')]

Final Response: Okay, let's calculate (12851 - 593) * 301 + 76.

First, let's calculate the difference between 12851 and 593:
12851 - 593 = 12258

Now, let's multiply 12258 by 301:
12258 * 30

"Okay, let's calculate (12851 - 593) * 301 + 76.\n\nFirst, let's calculate the difference between 12851 and 593:\n12851 - 593 = 12258\n\nNow, let's multiply 12258 by 301:\n12258 * 301 = 3,685,758\n\nFinally, let's add 76 to the result:\n3,685,758 + 76 = 3,685,834\n\nTherefore, the result of (12851 - 593) * 301 + 76 is 3,685,834."

In [18]:
chat_with_claude_tool(question2)


Initial Response:
Stop Reason: tool_use
Content: [TextBlock(citations=None, text='Here is the calculation:', type='text'), ToolUseBlock(id='toolu_01L3tsekypPpvAzbzRVnfmdP', input={'expression': '(12851 - 593) * 301 + 76'}, name='calculator', type='tool_use')]

Tool Used: calculator
Tool Input: {'expression': '(12851 - 593) * 301 + 76'}
Tool Result: 3689734
[TextBlock(citations=None, text='The result of the calculation (12851 - 593) * 301 + 76 is 3689734.', type='text')]

Final Response: The result of the calculation (12851 - 593) * 301 + 76 is 3689734.


'The result of the calculation (12851 - 593) * 301 + 76 is 3689734.'

In [35]:
eval('(12851 - 593) * 301 + 76')

3689734

In [19]:
question3 = "What is 15910385 divided by 193053?"

In [20]:
chat_with_claude_no_tool(question3)


Initial Response:
Stop Reason: end_turn
Content: [TextBlock(citations=None, text='To solve this problem, we need to divide 15910385 by 193053.\n\nThe step-by-step solution is as follows:\n\n15910385 ÷ 193053 = 82.5\n\nTo explain the calculation:\n- 193053 goes into 15910385 a total of 82 times (82 x 193053 = 15890336)\n- The remainder is 15910385 - 15890336 = 20049\n- 20049 ÷ 193053 = 0.5\n\nTherefore, the final answer is 82.5.', type='text')]
[TextBlock(citations=None, text='To solve this problem, we need to divide 15910385 by 193053.\n\nThe step-by-step solution is as follows:\n\n15910385 ÷ 193053 = 82.5\n\nTo explain the calculation:\n- 193053 goes into 15910385 a total of 82 times (82 x 193053 = 15890336)\n- The remainder is 15910385 - 15890336 = 20049\n- 20049 ÷ 193053 = 0.5\n\nTherefore, the final answer is 82.5.', type='text')]

Final Response: To solve this problem, we need to divide 15910385 by 193053.

The step-by-step solution is as follows:

15910385 ÷ 193053 = 82.5

To ex

'To solve this problem, we need to divide 15910385 by 193053.\n\nThe step-by-step solution is as follows:\n\n15910385 ÷ 193053 = 82.5\n\nTo explain the calculation:\n- 193053 goes into 15910385 a total of 82 times (82 x 193053 = 15890336)\n- The remainder is 15910385 - 15890336 = 20049\n- 20049 ÷ 193053 = 0.5\n\nTherefore, the final answer is 82.5.'

In [37]:
chat_with_claude_tool(question3)


Initial Response:
Stop Reason: tool_use
Content: [TextBlock(citations=None, text="Okay, let's calculate 15910385 divided by 193053 using the calculator tool.", type='text'), ToolUseBlock(id='toolu_01YKhdv2C1g4h3JTY1nCVCDN', input={'expression': '15910385 / 193053'}, name='calculator', type='tool_use')]

Tool Used: calculator
Tool Input: {'expression': '15910385 / 193053'}
Tool Result: 82.41459599177428
[TextBlock(citations=None, text='So, 15910385 divided by 193053 is 82.41459599177428.', type='text')]

Final Response: So, 15910385 divided by 193053 is 82.41459599177428.


'So, 15910385 divided by 193053 is 82.41459599177428.'

In [36]:
eval('15910385 / 193053')

82.41459599177428